# 물류 문서 RAG 전처리 파이프라인

PDF/스캔 물류 문서(B/L, Invoice, Packing List 등) → RAG 청크 변환 파이프라인

## 흐름

```
PDF/이미지
  ├─ [1] OCR              PaddleOCR PP-StructureV3 (레이아웃 + 표 + OCR)
  ├─ [2] 전처리            유니코드 정규화, 노이즈 제거 (표 보존)
  ├─ [3] 표 변환          HTML <table> → Markdown 표 (rowspan/colspan 전개)
  ├─ [4] 헤더 주입         물류 키워드 기반 인공 ##/### 헤더
  ├─ [5] 청킹              MarkdownHeaderTextSplitter → RecursiveCharacterTextSplitter
  └─ [6] 후처리            표 헤더 행 자동 복제
       ↓
  최종 RAG 청크 리스트 (content + metadata)
```

## 핵심 설계

| 문제 | 해결 |
|---|---|
| 본문 거의 전부가 `<table>` 블록 | HTML 표를 Markdown으로 변환, splitter가 행 단위로 자르게 함 |
| `#`/`##` 헤더가 없음 | 물류 도메인 키워드 21종을 정규식으로 탐지하여 인공 헤더 주입 |
| 셀 병합 정보 손실 | rowspan/colspan을 그리드로 전개해 동일 값 복제 |
| 큰 표 분할 시 헤더 행 손실 | 같은 section 첫 청크에서 헤더 행 추출 → 후속 청크에 prepend |
| 표 행 중간 절단 | RecursiveCharacterTextSplitter separator에 `\n|` 우선 지정 |


## 0. 의존성 설치

In [ ]:
# 시스템 의존성 (Ubuntu/Debian)
# !apt-get install -y poppler-utils

# Python 패키지
# !pip install paddleocr paddlepaddle pdf2image opencv-python
# !pip install beautifulsoup4 langchain-text-splitters
# GPU 사용 시: paddlepaddle 대신 paddlepaddle-gpu

## 1. OCR 모듈

PaddleOCR PP-StructureV3로 PDF/이미지에서 텍스트 + 표(HTML) 추출.

- **이미지 전처리**: 그레이스케일 → 노이즈 제거 → Adaptive Threshold → Deskew
- **레이아웃 정렬**: y좌표 20px 그리드로 묶어 위→아래, 좌→우 순서로 직렬화
- **표 출력**: `<table>...</table>` HTML 형태로 보존 (다음 단계에서 변환)
- **제목 마킹**: `[TITLE]` 프리픽스로 표시 → 헤더 주입 단계에서 활용

In [ ]:
from __future__ import annotations
from pathlib import Path
from typing import List, Dict, Any
import cv2
import numpy as np

from paddleocr import PPStructureV3
from pdf2image import convert_from_path


class LogisticsOCR:
    """물류 문서 OCR 처리기"""

    def __init__(self, lang: str = "korean", use_gpu: bool = False, dpi: int = 300):
        self.engine = PPStructureV3(lang=lang, use_gpu=use_gpu, show_log=False)
        self.dpi = dpi

    @staticmethod
    def _preprocess_image(img: np.ndarray) -> np.ndarray:
        """OCR 정확도 향상을 위한 이미지 보정"""
        if len(img.shape) == 3:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        else:
            gray = img

        # 노이즈 제거
        denoised = cv2.fastNlMeansDenoising(gray, h=10)

        # Adaptive Threshold (조명 불균일 대응)
        binary = cv2.adaptiveThreshold(
            denoised, 255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY, 31, 15,
        )

        # 기울기 보정 (deskew)
        coords = np.column_stack(np.where(binary < 255))
        if len(coords) > 0:
            angle = cv2.minAreaRect(coords)[-1]
            angle = -(90 + angle) if angle < -45 else -angle
            if abs(angle) > 0.5:
                (h, w) = binary.shape[:2]
                M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
                binary = cv2.warpAffine(
                    binary, M, (w, h),
                    flags=cv2.INTER_CUBIC,
                    borderMode=cv2.BORDER_REPLICATE,
                )

        return cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR)

    def _pdf_to_images(self, pdf_path: Path) -> List[np.ndarray]:
        pil_pages = convert_from_path(str(pdf_path), dpi=self.dpi)
        return [cv2.cvtColor(np.array(p), cv2.COLOR_RGB2BGR) for p in pil_pages]

    @staticmethod
    def _serialize_page(page_result: List[Dict[str, Any]]) -> str:
        """페이지 결과를 위→아래, 좌→우 순으로 직렬화"""
        blocks = sorted(
            page_result,
            key=lambda b: (b["bbox"][1] // 20, b["bbox"][0]),
        )

        parts = []
        for blk in blocks:
            btype = blk.get("type", "").lower()
            if btype == "table":
                html = blk.get("res", {}).get("html", "")
                if html:
                    parts.append(html)
            elif btype in ("title", "text", "list", "header", "footer"):
                lines = blk.get("res", [])
                if isinstance(lines, list):
                    text = "\n".join(
                        ln.get("text", "") for ln in lines if isinstance(ln, dict)
                    )
                else:
                    text = str(lines)
                if text.strip():
                    if btype == "title":
                        parts.append(f"\n[TITLE] {text.strip()}\n")
                    else:
                        parts.append(text.strip())

        return "\n\n".join(parts)

    def extract(self, file_path) -> str:
        """PDF/이미지에서 텍스트 + HTML 테이블 추출"""
        file_path = Path(file_path)

        if file_path.suffix.lower() == ".pdf":
            images = self._pdf_to_images(file_path)
        else:
            img = cv2.imread(str(file_path))
            if img is None:
                raise ValueError(f"이미지를 읽을 수 없습니다: {file_path}")
            images = [img]

        page_texts = []
        for idx, img in enumerate(images):
            pre = self._preprocess_image(img)
            result = self.engine(pre)
            page_text = self._serialize_page(result)
            page_texts.append(f"<!-- PAGE {idx + 1} -->\n{page_text}")

        return "\n\n".join(page_texts)


## 2. 전처리 모듈

OCR 결과 텍스트 정제. **표(`<table>`)는 건드리지 않고 본문만** 정규화.

- 유니코드 NFC 정규화 (한글 자모 결합)
- 제어문자 제거 (탭/줄바꿈은 보존)
- 줄 끝 하이픈 이어붙임 (`abc-\ndef` → `abcdef`)
- 한글 사이 잘못된 공백 제거
- 페이지 번호 패턴 제거

In [ ]:
import re
import unicodedata

_TABLE_PATTERN = re.compile(r"<table[^>]*>.*?</table>", re.DOTALL | re.IGNORECASE)


def _clean_plain_text(text: str) -> str:
    """본문 텍스트(표 제외) 정제"""
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"[\x00-\x08\x0b-\x0c\x0e-\x1f]", "", text)
    text = re.sub(r"-\n(\w)", r"\1", text)
    text = re.sub(r"(?<=[가-힣])\s+(?=[가-힣])", "", text)
    text = re.sub(r"^\s*-?\s*\d{1,3}\s*-?\s*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def clean_ocr_output(raw: str) -> str:
    """<table> 블록은 원형 보존, 나머지 본문만 정제"""
    parts = []
    last_end = 0

    for m in _TABLE_PATTERN.finditer(raw):
        before = raw[last_end:m.start()]
        if before.strip():
            parts.append(_clean_plain_text(before))
        parts.append(m.group(0))
        last_end = m.end()

    tail = raw[last_end:]
    if tail.strip():
        parts.append(_clean_plain_text(tail))

    return "\n\n".join(parts)


## 3. 표 변환 모듈 (HTML → Markdown)

PaddleOCR이 출력한 HTML 표를 splitter 친화적인 Markdown 표로 변환.

**핵심 처리**:
- `rowspan`/`colspan`을 그리드로 **전개**해 병합 셀 값을 인접 셀에 복제 (RAG 검색 시 행 단독으로도 의미 보존)
- 셀 내부 줄바꿈/파이프 이스케이프
- 표 앞뒤 빈 줄 추가 → `\n\n` separator로 분할 가능

In [ ]:
from bs4 import BeautifulSoup
from typing import List


def _clean_cell_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    return s.replace("|", "\\|")


def _expand_table_to_grid(table_html: str) -> List[List[str]]:
    """
    HTML 테이블을 rowspan/colspan을 전개한 2D 그리드로 변환.
    병합 셀은 동일 값을 인접 위치에 복제.
    """
    soup = BeautifulSoup(table_html, "html.parser")
    rows = soup.find_all("tr")
    occupied = {}

    for r_idx, row in enumerate(rows):
        cells = row.find_all(["td", "th"])
        c_idx = 0
        for cell in cells:
            # 위쪽 rowspan이 차지한 칸 스킵
            while (r_idx, c_idx) in occupied:
                c_idx += 1

            text = _clean_cell_text(cell.get_text(separator=" "))
            try:
                rowspan = int(cell.get("rowspan", 1))
            except (TypeError, ValueError):
                rowspan = 1
            try:
                colspan = int(cell.get("colspan", 1))
            except (TypeError, ValueError):
                colspan = 1

            for dr in range(rowspan):
                for dc in range(colspan):
                    occupied[(r_idx + dr, c_idx + dc)] = text
            c_idx += colspan

    if not occupied:
        return []

    max_r = max(r for r, _ in occupied)
    max_c = max(c for _, c in occupied)
    return [
        [occupied.get((r, c), "") for c in range(max_c + 1)]
        for r in range(max_r + 1)
    ]


def html_table_to_markdown(table_html: str) -> str:
    grid = _expand_table_to_grid(table_html)
    if not grid:
        return ""

    n_cols = len(grid[0])
    md_lines = []
    md_lines.append("| " + " | ".join(grid[0]) + " |")
    md_lines.append("|" + "|".join(["---"] * n_cols) + "|")
    for row in grid[1:]:
        md_lines.append("| " + " | ".join(row) + " |")

    return "\n" + "\n".join(md_lines) + "\n"


def convert_all_tables(text: str) -> str:
    """본문 내 모든 <table>을 Markdown 표로 치환"""
    return _TABLE_PATTERN.sub(
        lambda m: html_table_to_markdown(m.group(0)),
        text,
    )


## 4. 헤더 주입 모듈

물류 문서는 마크다운 헤더(`#`, `##`)가 없으므로 **키워드 기반으로 인공 헤더 주입**.

- 문서 최상단: `# {doc_type}` (B/L, Invoice 등 자동 감지)
- 본문 내 키워드 매칭 라인 앞: `## {정규화 헤더명}`
- 같은 헤더 중복 방지 (첫 매칭만 헤더화)
- Markdown 표 행(`| ... |`)은 헤더화 제외

**커버 키워드 21종**: Shipper, Consignee, Notify Party, Vessel, POL, POD, Description of Goods, Gross Weight, HS Code, Container No, Seal No 등

In [ ]:
from typing import Tuple


LOGISTICS_SECTION_PATTERNS: List[Tuple[re.Pattern, int, str]] = [
    (re.compile(r"\b(shipper|consignor|송하인|수출자)\b", re.I), 2, "Shipper"),
    (re.compile(r"\b(consignee|수하인|수입자)\b", re.I), 2, "Consignee"),
    (re.compile(r"\b(notify\s*party|통지처|notify)\b", re.I), 2, "Notify Party"),
    (re.compile(r"\b(vessel|voyage|선박|항차|vsl)\b", re.I), 2, "Vessel & Voyage"),
    (re.compile(r"\b(port\s*of\s*loading|선적항|POL)\b", re.I), 2, "Port of Loading"),
    (re.compile(r"\b(port\s*of\s*discharge|양륙항|POD)\b", re.I), 2, "Port of Discharge"),
    (re.compile(r"\b(place\s*of\s*delivery|인도지|최종도착지)\b", re.I), 2, "Place of Delivery"),
    (re.compile(r"\b(description\s*of\s*goods|품명|상품\s*내역|goods\s*description)\b", re.I), 2, "Description of Goods"),
    (re.compile(r"\b(marks\s*(and|&)\s*numbers|화인|shipping\s*marks)\b", re.I), 2, "Marks and Numbers"),
    (re.compile(r"\b(gross\s*weight|총중량|G\.?W\.?)\b", re.I), 2, "Gross Weight"),
    (re.compile(r"\b(net\s*weight|순중량|N\.?W\.?)\b", re.I), 2, "Net Weight"),
    (re.compile(r"\b(measurement|용적|CBM|M3)\b", re.I), 2, "Measurement"),
    (re.compile(r"\b(freight|운임|charges|freight\s*charges)\b", re.I), 2, "Freight & Charges"),
    (re.compile(r"\b(HS\s*CODE|품목분류|HSK)\b", re.I), 2, "HS Code"),
    (re.compile(r"\b(invoice\s*(no|number|#)|송장\s*번호|inv\s*no)\b", re.I), 2, "Invoice Info"),
    (re.compile(r"\b(packing\s*list|포장\s*명세|P/L)\b", re.I), 2, "Packing List"),
    (re.compile(r"\b(bill\s*of\s*lading|선하증권|B/L\s*no)\b", re.I), 2, "B/L Info"),
    (re.compile(r"\b(container\s*(no|number)|컨테이너\s*번호)\b", re.I), 2, "Container"),
    (re.compile(r"\b(seal\s*(no|number)|봉인\s*번호)\b", re.I), 2, "Seal Number"),
    (re.compile(r"\b(terms\s*of\s*(payment|delivery)|결제조건|인도조건|incoterms)\b", re.I), 2, "Terms"),
    (re.compile(r"\b(country\s*of\s*origin|원산지)\b", re.I), 2, "Country of Origin"),
]


def _detect_doc_type(text: str) -> str:
    """문서 유형 추정"""
    head = text[:1500].lower()
    if re.search(r"bill\s*of\s*lading|선하증권", head):
        return "BILL OF LADING"
    if re.search(r"commercial\s*invoice|상업\s*송장", head):
        return "COMMERCIAL INVOICE"
    if re.search(r"packing\s*list|포장\s*명세", head):
        return "PACKING LIST"
    if re.search(r"certificate\s*of\s*origin|원산지\s*증명", head):
        return "CERTIFICATE OF ORIGIN"
    return "LOGISTICS DOCUMENT"


def inject_headers(text: str, doc_type: str = None) -> str:
    """텍스트에 마크다운 헤더 주입"""
    if doc_type is None:
        doc_type = _detect_doc_type(text)

    lines = text.split("\n")
    output = [f"# {doc_type}", ""]
    used = set()
    in_table = False

    for line in lines:
        stripped = line.strip()

        # Markdown 표 행 감지
        is_table_row = stripped.startswith("|") and stripped.endswith("|")
        if is_table_row:
            in_table = True
            output.append(line)
            continue
        elif in_table and not is_table_row:
            in_table = False

        # [TITLE] 마킹 처리
        if stripped.startswith("[TITLE]"):
            title_text = stripped.replace("[TITLE]", "").strip()
            output.append("")
            output.append(f"## {title_text}")
            continue

        # 키워드 기반 헤더 매칭
        matched = None
        for pattern, level, name in LOGISTICS_SECTION_PATTERNS:
            if name in used:
                continue
            if pattern.search(line):
                matched = (level, name)
                used.add(name)
                break

        if matched:
            level, name = matched
            output.append("")
            output.append(f"{'#' * level} {name}")
            output.append(line)
        else:
            output.append(line)

    return "\n".join(output)


## 5. 청킹 모듈 (LangChain Splitter 2단)

**1단**: `MarkdownHeaderTextSplitter` — 의미 단위 분할, 헤더 정보를 메타데이터에 저장
**2단**: `RecursiveCharacterTextSplitter` — 길이 정규화

**Separator 우선순위**: 헤더 → 빈 줄 → 표 행 시작(`\n|`) → 줄바꿈 → 문장 종결 → 공백

**오버랩 권장값** (한국어 + 표 위주):
| 임베딩 모델 | chunk_size | chunk_overlap | 비율 |
|---|---|---|---|
| bge-m3 | 1024 | 128 | ~12% |
| multilingual-e5-large | 512 | 80 | ~15% |
| ko-sroberta | 450 | 60 | ~13% |

표가 많을수록 오버랩을 약간 높게(15%) 잡아 행 손실 방지.

In [ ]:
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

DEFAULT_CHUNK_SIZE = 512
DEFAULT_CHUNK_OVERLAP = 80   # ≈15%

TABLE_ROW_RE = re.compile(r"^\s*\|.+\|\s*$")
TABLE_SEP_RE = re.compile(r"^\s*\|[\s\-:|]+\|\s*$")


def _split_by_headers(text: str):
    """1단: 헤더 기반 분할"""
    splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=[
            ("#", "doc_type"),
            ("##", "section"),
            ("###", "subsection"),
        ],
        strip_headers=False,
    )
    docs = splitter.split_text(text)
    return [
        {"content": d.page_content, "metadata": dict(d.metadata)}
        for d in docs
    ]


def _split_by_length(text: str, chunk_size: int, chunk_overlap: int):
    """2단: 길이 기반 분할 (표 행 경계 우선)"""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=[
            "\n### ",
            "\n## ",
            "\n# ",
            "\n\n",
            "\n|",     # 표 행 경계 우선
            "\n",
            "。",
            ". ",
            " ",
            "",
        ],
    )
    return splitter.split_text(text)


def _extract_table_header(text: str):
    """청크 내 표 헤더 행 + 구분선 추출"""
    lines = text.split("\n")
    for i, ln in enumerate(lines):
        if TABLE_ROW_RE.match(ln) and i + 1 < len(lines) and TABLE_SEP_RE.match(lines[i + 1]):
            return f"{ln}\n{lines[i + 1]}"
    return None


def _has_table_body_only(text: str) -> bool:
    """헤더 행/구분선 없이 표 데이터 행만 포함하는지 검사"""
    lines = [ln for ln in text.split("\n") if ln.strip()]
    table_lines = [ln for ln in lines if TABLE_ROW_RE.match(ln)]
    if not table_lines:
        return False
    has_sep = any(TABLE_SEP_RE.match(ln) for ln in lines)
    return len(table_lines) >= 1 and not has_sep


def _repeat_table_headers(chunks):
    """잘려나간 표 청크 앞에 헤더 행 자동 복제"""
    section_header = {}
    for ch in chunks:
        section = ch["metadata"].get("section", "_root")
        hdr = _extract_table_header(ch["content"])
        if hdr and section not in section_header:
            section_header[section] = hdr

    for ch in chunks:
        section = ch["metadata"].get("section", "_root")
        if _has_table_body_only(ch["content"]) and section in section_header:
            ch["content"] = section_header[section] + "\n" + ch["content"]
            ch["metadata"]["table_header_repeated"] = True
    return chunks


def chunk_document(text: str,
                   chunk_size: int = DEFAULT_CHUNK_SIZE,
                   chunk_overlap: int = DEFAULT_CHUNK_OVERLAP):
    """헤더 주입까지 끝난 텍스트 → RAG 청크 리스트"""
    header_chunks = _split_by_headers(text)

    final_chunks = []
    for hc in header_chunks:
        sub_texts = _split_by_length(hc["content"], chunk_size, chunk_overlap)
        for idx, st in enumerate(sub_texts):
            meta = dict(hc["metadata"])
            meta["sub_index"] = idx
            final_chunks.append({"content": st, "metadata": meta})

    final_chunks = _repeat_table_headers(final_chunks)

    for i, ch in enumerate(final_chunks):
        ch["metadata"]["chunk_id"] = i

    return final_chunks


## 6. 통합 파이프라인 클래스

위 모든 모듈을 하나의 클래스로 통합.

In [ ]:
class LogisticsRAGPipeline:
    def __init__(self,
                 lang: str = "korean",
                 use_gpu: bool = False,
                 dpi: int = 300,
                 chunk_size: int = DEFAULT_CHUNK_SIZE,
                 chunk_overlap: int = DEFAULT_CHUNK_OVERLAP):
        self.ocr = LogisticsOCR(lang=lang, use_gpu=use_gpu, dpi=dpi)
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    def process(self, file_path, doc_type=None, return_intermediate=False):
        # 1) OCR
        raw_text = self.ocr.extract(file_path)

        # 2) 전처리
        cleaned = clean_ocr_output(raw_text)

        # 3) HTML 표 → Markdown
        with_md_tables = convert_all_tables(cleaned)

        # 4) 인공 헤더 주입
        structured = inject_headers(with_md_tables, doc_type=doc_type)

        # 5) 청킹
        chunks = chunk_document(structured,
                                chunk_size=self.chunk_size,
                                chunk_overlap=self.chunk_overlap)

        for ch in chunks:
            ch["metadata"]["source"] = str(file_path)

        result = {"chunks": chunks}
        if return_intermediate:
            result["intermediate"] = {
                "raw_ocr": raw_text,
                "cleaned": cleaned,
                "markdown_tables": with_md_tables,
                "structured": structured,
            }
        return result


## 7. 검증 (OCR 제외, 모의 출력으로 단계별 확인)

실제 PaddleOCR이 없는 환경에서도 후속 파이프라인을 검증할 수 있도록,
PaddleOCR PP-StructureV3가 만들어낼 법한 출력을 모사한 샘플로 테스트.

In [ ]:
MOCK_OCR_OUTPUT = """<!-- PAGE 1 -->
[TITLE] BILL OF LADING

B/L No: HJSCKR1234567
Shipper: HYUNDAI MERCHANT MARINE CO.,LTD.
        SEOUL, KOREA
Consignee: ABC TRADING LLC
          123 MAIN ST, LOS ANGELES, CA 90001, USA
Notify Party: SAME AS CONSIGNEE

Vessel: HMM ALGECIRAS  Voyage: 0123E
Port of Loading: BUSAN, KOREA
Port of Discharge: LOS ANGELES, USA
Place of Delivery: LOS ANGELES CY

<table><tr><td>Marks and Numbers</td><td>Description of Goods</td><td rowspan="2">Gross Weight (KGS)</td><td rowspan="2">Measurement (CBM)</td></tr><tr><td>SHIPPING MARKS</td><td>품명 / 수량</td></tr><tr><td>ABC-001</td><td>ELECTRONIC PARTS  500 CTNS</td><td>12,500.00</td><td>45.230</td></tr><tr><td>ABC-002</td><td>PLASTIC RESIN  300 BAGS</td><td>7,800.00</td><td>22.100</td></tr><tr><td>ABC-003</td><td>STEEL PIPES  100 BUNDLES</td><td>25,400.00</td><td>38.500</td></tr><tr><td colspan="2">TOTAL</td><td>45,700.00</td><td>105.830</td></tr></table>

Freight & Charges: PREPAID
Container No: HMMU1234567 / 40HC
Seal No: KR98765

<!-- PAGE 2 -->
HS Code: 8542.31-0000
Country of Origin: REPUBLIC OF KOREA
Terms of Payment: T/T 30 DAYS AFTER B/L DATE

<table><tr><td>Container No</td><td>Seal No</td><td>Size/Type</td><td>Packages</td><td>Weight</td></tr><tr><td>HMMU1234567</td><td>KR98765</td><td>40HC</td><td>500 CTNS</td><td>12,500.00 KGS</td></tr><tr><td>HMMU7654321</td><td>KR98766</td><td>40HC</td><td>400 CTNS</td><td>33,200.00 KGS</td></tr></table>

- 1 -
"""

print(f"원본 길이: {len(MOCK_OCR_OUTPUT)}자")
print(MOCK_OCR_OUTPUT[:400])


### 7-1. 전처리 결과

In [ ]:
cleaned = clean_ocr_output(MOCK_OCR_OUTPUT)
print(cleaned[:600])
print(f"\n[길이] raw={len(MOCK_OCR_OUTPUT)} → cleaned={len(cleaned)}")
assert "<table>" in cleaned, "표 보존 OK"


### 7-2. HTML 표 → Markdown 변환 결과

In [ ]:
with_md = convert_all_tables(cleaned)
print(with_md[:1500])

# 검증
assert "<table>" not in with_md
assert with_md.count("Gross Weight (KGS)") >= 2, "rowspan 전개 OK"
assert with_md.count("TOTAL") >= 2, "colspan 전개 OK"
print("\n✓ rowspan/colspan 전개 정상")


### 7-3. 헤더 주입 결과

In [ ]:
structured = inject_headers(with_md)
print(structured[:1800])

assert structured.startswith("# BILL OF LADING"), "doc_type 자동 감지 OK"
assert "## Shipper" in structured
assert "## Consignee" in structured
assert "## Port of Loading" in structured
print("\n✓ doc_type 자동 감지 + 키워드 헤더 주입 정상")


### 7-4. 최종 청크

In [ ]:
chunks = chunk_document(structured, chunk_size=512, chunk_overlap=80)
print(f"총 청크 수: {len(chunks)}\n")

for ch in chunks:
    print(f"[chunk_id={ch['metadata']['chunk_id']}] "
          f"section={ch['metadata'].get('section', '-')} "
          f"len={len(ch['content'])}")


In [ ]:
# 표가 포함된 청크 자세히 보기
for ch in chunks:
    if "|" in ch["content"] and "---" in ch["content"]:
        print(f"=== Chunk {ch['metadata']['chunk_id']} (section: {ch['metadata'].get('section')}) ===")
        print(ch["content"])
        print()


## 8. 실제 PDF 처리 예시

PaddleOCR이 설치된 환경에서 실제 PDF 파일을 처리하는 코드.
(샘플 파일을 준비한 뒤 경로만 바꿔서 실행)

In [ ]:
# 실제 PDF/이미지 처리 (PaddleOCR 설치 필요)
#
# pipe = LogisticsRAGPipeline(
#     lang="korean",
#     use_gpu=False,
#     chunk_size=512,
#     chunk_overlap=80,
# )
#
# result = pipe.process("samples/bl_001.pdf", return_intermediate=True)
#
# print(f"총 청크 수: {len(result['chunks'])}")
# for ch in result["chunks"][:5]:
#     print(ch["metadata"])
#     print(ch["content"][:150])
#     print("---")
#
# # 중간 산출물 확인 (디버깅용)
# print(result["intermediate"]["raw_ocr"][:500])


## 다음 단계

여기까지가 **OCR → 전처리 → 청킹 + 오버랩** 파이프라인.
이어서 필요한 작업:

1. **임베딩 + 벡터스토어 연결**
   - 한국어 추천: `BAAI/bge-m3` (1024차원, 8K 컨텍스트) 또는 `intfloat/multilingual-e5-large`
   - 벡터스토어: FAISS, Chroma, Qdrant 등

2. **하이브리드 검색**
   - 물류 문서는 컨테이너번호·HS코드·B/L 번호 같은 정확 매칭 키워드가 많음
   - BM25 + 벡터 검색 병행 추천 (`langchain.retrievers.EnsembleRetriever`)

3. **`LOGISTICS_SECTION_PATTERNS` 확장**
   - 실제 처리할 문서 유형(D/O, A/N, C/O 등)의 키워드 보강
   - 사내 문서 양식에 특화된 정규식 추가
